In [1]:
import pandas as pd
import numpy as np

In [2]:
deals_raw = pd.read_excel('Deals (Done).xlsx')
deals = deals_raw.copy()

In [3]:
deals.head()

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Successful Deal,Contact Name,City,Level of Deutsch
0,5.805028e+18,Ben Hall,NaN,NaN,New Lead,NaN,/eng/test,03.07.23women,NaN,v16,...,NaN,21.06.2024 15:30,NaN,NaN,NaN,NaN,0.0,5.805028e+18,NaN,NaN
1,5.805028e+18,Ulysses Adams,NaN,NaN,New Lead,NaN,/at-eng,NaN,NaN,NaN,...,Morning,21.06.2024 15:23,6.0,NaN,0,2000,0.0,5.805028e+18,NaN,NaN
2,5.805028e+18,Ulysses Adams,21.06.2024,D - Non Target,Lost,Non target,/at-eng,engwien_AT,0:26:43,b1-at,...,NaN,21.06.2024 14:45,NaN,NaN,NaN,NaN,0.0,5.805028e+18,NaN,NaN
3,5.805028e+18,Eva Kent,21.06.2024,E - Non Qualified,Lost,Invalid number,/eng,04.07.23recentlymoved_DE,1:00:04,bloggersvideo14com,...,NaN,21.06.2024 13:32,NaN,NaN,NaN,NaN,0.0,5.805028e+18,NaN,NaN
4,5.805028e+18,Ben Hall,21.06.2024,D - Non Target,Lost,Non target,/eng,discovery_DE,0:53:12,website,...,NaN,21.06.2024 13:21,NaN,NaN,NaN,NaN,0.0,5.805028e+18,NaN,NaN


In [4]:
# Vollständige Zeilenduplikate entfernen
deals = deals.drop_duplicates()

# Zeilen ohne jegliche Daten entfernen
deals = deals.replace(r'^\s*$', np.nan, regex=True).replace('#REF!', np.nan)
deals = deals.dropna(how='all')

### Konvertierung von Zeitdaten und Auffüllen von Fehlwerten im Schulungstyp (oder Zeitdaten konvertieren und Fehlwerte beim Schulungstyp auffüllen)

In [5]:

# 1. Umwandlung von Daten in das datetime-Format
deals['Created Time'] = pd.to_datetime(deals['Created Time'], dayfirst=True)
deals['Closing Date'] = pd.to_datetime(deals['Closing Date'], dayfirst=True)

deals['Created Date'] = deals['Created Time'].dt.normalize()
deals['Day of week'] = deals['Created Time'].dt.day_name()


hour = deals['Created Time'].dt.hour

conditions = [
    (hour >= 5) & (hour < 12),
    (hour >= 12) & (hour < 18),
    (hour >= 18) & (hour < 22)
]
choices = ['Morning', 'Day', 'Evening']

deals['Day_Period'] = np.select(conditions, choices, default='Night')
deals['Day_Period'] = deals['Day_Period'].astype('category')

# 4. Behandlung von Fehlwerten beim Schulungstyp (Education Type)
# Einfache Logik (Morgen/Abend) zum Auffüllen verwenden:
# Wenn die Uhrzeit im Intervall 05:00–15:00 Uhr liegt -> Morning, ansonsten -> Evening
temp_fill_logic = np.where((hour >= 5) & (hour < 15), 'Morning', 'Evening')

deals['Education Type'] = deals['Education Type'].fillna(pd.Series(temp_fill_logic, index=deals.index))


print(f"Fehlwerte beim Schulungstyp: {deals['Education Type'].isna().sum()}")

print("\nVerteilung der Schulungstypen:")
print(deals['Education Type'].value_counts())

print("\nVerteilung nach Tageszeiten (Day_Period):")
print(deals['Day_Period'].value_counts())

Fehlwerte beim Schulungstyp: 0

Verteilung der Schulungstypen:
Education Type
Morning    11064
Evening    10526
Name: count, dtype: int64

Verteilung nach Tageszeiten (Day_Period):
Day_Period
Day        7537
Morning    5818
Evening    5151
Night      3084
Name: count, dtype: int64


In [6]:
# 1. Wenn die Differenz negativ ist (innerhalb eines Tages), 1 Tag hinzufügen
diff = (deals['Closing Date'] - deals['Created Time']).dt.total_seconds()
mask_small = diff < 0
deals.loc[mask_small, 'Closing Date'] = deals['Closing Date'] + pd.Timedelta(days=1)

deals['Cycle_Time_Hours'] = (deals['Closing Date'] - deals['Created Time']).dt.total_seconds() / 3600

deals.loc[deals['Cycle_Time_Hours'] < 0, 'Cycle_Time_Hours'] = 0


deals['Cycle_Time_Hours'] = deals['Cycle_Time_Hours'].round(1)


print("Anzahl der negativen Werte")
print((deals['Cycle_Time_Hours'] < 0).sum())

Anzahl der negativen Werte
0


In [7]:
deals[deals['Cycle_Time_Hours'] < 0]

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Initial Amount Paid,Offer Total Amount,Successful Deal,Contact Name,City,Level of Deutsch,Created Date,Day of week,Day_Period,Cycle_Time_Hours


### # Verarbeitung von Sprachniveaus (Level Language)

In [8]:
for col in ['Initial Amount Paid', 'Offer Total Amount']:
    deals[col] = deals[col].astype(str)
    # # Alles Überflüssige entfernen; nur Ziffern, Punkte und Minuszeichen beibehalten
    deals[col] = deals[col].str.replace(',', '.').str.replace(r'[^0-9.\-]', '', regex=True)
    deals[col] = pd.to_numeric(deals[col], errors='coerce')

In [9]:
import re

def final_clean_level(text):
    # 1. # Zuerst in String umwandeln und Schreibweise/Tastaturlayout bereinigen
    text = str(text).upper().strip()
    rep = {"А": "A", "В": "B", "С": "C", "Б": "B"}
    for rus, eng in rep.items():
        text = text.replace(rus, eng)
    
    # 2. # Nach der Standardkombination Buchstabe + Zahl suchen (A1, B2 usw.)
    match = re.search(r'[ABC][012]', text)
    
    if match:
        found_level = match.group(0)
        
        # # Ersetzungsregeln anwenden:
        mapping = {
            'A0': 'A1',
            'F2': 'A2'
        }
        return mapping.get(found_level, found_level)
    
    # 3. # Einzelne Buchstaben verarbeiten
    if text == 'C':
        return 'C1'
    if text == 'A':
        return 'A1'
    
   
    return 'Unknown'

deals['Level of Deutsch'] = deals['Level of Deutsch'].apply(final_clean_level)

print("Finale Verteilung der Sprachniveaus:")
print(deals['Level of Deutsch'].value_counts())

Finale Verteilung der Sprachniveaus:
Level of Deutsch
Unknown    20390
B1           817
B2           171
A2           149
A1            32
C1            28
C2             3
Name: count, dtype: int64


### # Verarbeitung von Städteangaben (City)

In [10]:
# 1. # Zuerst in Kleinbuchstaben umwandeln und Leerzeichen an den Rändern entfernen
deals['City'] = deals['City'].astype(str).str.strip()

# 2. # Massenersatz von fehlerhaften Werten durch NaN (zur späteren Auffüllung als "Unknown")
garbage_values = ['-', '.', 'nan', 'Unknown']
deals.loc[deals['City'].isin(garbage_values), 'City'] = np.nan

# 3. # Adressverarbeitung (falls ein Komma vorhanden ist, alles DAVOR nehmen)
deals['City'] = deals['City'].str.split(',').str[0]

# 4. # Postleitzahlen entfernen (5 Ziffern am Anfang der Zeile)
deals['City'] = deals['City'].str.replace(r'^\d{5}\s+', '', regex=True)

deals['City'] = deals['City'].str.replace('‑', '-')
deals['City'] = deals['City'].str.title().fillna('Unknown')
deals['Quality'] = deals[ 'Quality'].fillna('Not Defined')


### # Korrektur der Summe von Initial Amount und Offer Total

In [11]:
# # Erstellen einer Maske (Bedingung): wo der bezahlte Betrag größer ist als der Gesamtbetrag
mask = deals['Initial Amount Paid'] > deals['Offer Total Amount']

temp_paid = deals.loc[mask, 'Initial Amount Paid'].copy()

# # # Werte untereinander tauschen
deals.loc[mask, 'Initial Amount Paid'] = deals.loc[mask, 'Offer Total Amount']
deals.loc[mask, 'Offer Total Amount'] = temp_paid

print(f"# Korrigiert: {mask.sum()}")

# Korrigiert: 55


In [12]:
# Auffüllen von Fehlwerten bei den Schulungsmonaten
deals['Months of study'] = deals[ 'Months of study'].fillna(-1).astype(int)

### Auffüllen von Product

In [13]:
# # Erstellen eines Zuordnungswörterbuchs: Was in "Page" gesucht wird -> Was in "Product" eingetragen wird
mapping = {
    'digital-marketing': 'Digital Marketing',
    'ux-ui': 'UX/UI Design',
    'web-developer': 'Web Developer',
    'test': 'Lead Magnet (Test)',
    'workshop': 'Lead Magnet (Workshop)',
    'webinar': 'Lead Magnet (Webinar)'
}

# # Schleife zur Iteration über das Dictionary und zum Auffüllen der fehlenden Werte (NaN)
for key, value in mapping.items():
    deals.loc[deals['Product'].isna() & deals['Page'].str.contains(key, na=False), 'Product'] = value

print(f"# Verbleibende Fehlwerte in Product: {deals['Product'].isna().sum()}")

# Verbleibende Fehlwerte in Product: 8217


In [ ]:
## Erstellen einer neuen Datei

In [18]:
deals.to_csv('Deals_Final.csv', index=False)